# YOLO Segmentation — RGB Tile Preprocessing

Prepares the RGB segmentation dataset for YOLOv11-seg training.

**Pipeline**
1. Load raw RGB GeoTIFF tiles, tree-crown polygons, and a study-area boundary.
2. Convert each tile to uint8 in memory.
3. Slide a 512 × 512 window; skip patches outside the study-area boundary.
4. For each valid patch: clip crown polygons to the patch extent, convert
   clipped polygons to YOLO segmentation format, save image + label files.
5. Background patches sub-sampled to 10 % for training; val/test positive only.


**Import libraries**

In [ ]:
import math
import os
import random
import shutil
import sys
from collections import Counter
from pathlib import Path
from typing import Dict, List

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.windows import Window
from shapely.geometry import box
from shapely.validation import make_valid

# Locate project root (works from any sub-directory inside the repo)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.paths import RAW_DIR, EXTERNAL_DIR, INTERIM_DIR, PROCESSED_DIR


In [ ]:
random.seed(42)

**Define paths**

In [ ]:
# Cleaned tree-crown polygons (one row per individual crown)
YOLO_SEGMENTATION_GROUND_TRUTH = (
    INTERIM_DIR
    / "yolo_segmentation_gt_prep"
    / "enschede_yolo_segmentation_ground_truth_v01.shp"
)

# Directory that contains the raw RGB GeoTIFF tiles
RGB_TILES_DIR = RAW_DIR / "rgb_tiles"

# Output root for train/val/test splits
SPLITS_DIR = PROCESSED_DIR / "splits_rgb_seg"


**Define constants**

In [ ]:
# Raster tile IDs — one per geographic area, mapped to a train/val/test split
RASTER_IDS = ["34FN2", "34FZ2", "35AN1"]

# Square patch size in pixels
TILE_SIZE = 512

# Single class: every annotated polygon is a tree crown
TREE_CLASS_ID = 0

# Fractional overlap between adjacent patches, per split.
# Segmentation uses 0 % overlap for all splits — unlike detection, the model
# sees the full polygon shape and does not benefit from repeated views of the
# same crown at different offsets.
OVERLAP_BY_SPLIT = {
    "train": 0.0,
    "val":   0.0,
    "test":  0.0,
}

# Which geographic tile belongs to which split
SPLIT_MAP = {
    "34FN2": "train",
    "34FZ2": "val",
    "35AN1": "test",
}

# Fraction of background (no-crown) patches kept for training
BACKGROUND_RATIO = 0.10


**Helper: convert band to uint8**

In [ ]:
def convert_to_uint8(band: np.ndarray) -> np.ndarray:
    """
    Normalise a single raster band to the 0-255 uint8 range.

    The band is first linearly rescaled so that its minimum maps to 0 and its
    maximum maps to 1, then multiplied by 255 and cast to uint8.  This makes
    the dynamic range of different tiles comparable regardless of their
    original radiometric values.

    Args:
        band: 2-D NumPy array of any numeric dtype.

    Returns:
        2-D NumPy array of dtype uint8.
    """
    band = (band - band.min()) / (band.max() - band.min())
    return (band * 255).astype("uint8")


**Stack RGB bands in memory**

In [ ]:
def stack_rgb(rgb_path: str) -> tuple:
    """
    Open an RGB GeoTIFF, convert each band to uint8, and return the stacked
    array together with the rasterio dataset metadata.

    The original tiles may be stored as float32 or uint16; converting to uint8
    reduces memory usage and is the format expected by YOLO training pipelines.

    Args:
        rgb_path: File path to a 3-band RGB GeoTIFF.

    Returns:
        stacked: NumPy array of shape (3, H, W) and dtype uint8 (R, G, B).
        meta:    Rasterio metadata dict for the source file (used when writing
                 patch GeoTIFFs so that spatial reference is preserved).
        src_obj: The open rasterio DatasetReader (kept open so the caller can
                 pass it directly to the tiling functions without re-opening).
    """
    src_obj = rasterio.open(rgb_path)
    R = convert_to_uint8(src_obj.read(1))
    G = convert_to_uint8(src_obj.read(2))
    B = convert_to_uint8(src_obj.read(3))
    stacked = np.stack([R, G, B], axis=0)
    meta = src_obj.meta.copy()
    meta.update({"driver": "GTiff", "count": 3, "dtype": "uint8"})
    return stacked, meta, src_obj


**Generate YOLO segmentation label for one crown**

In [ ]:
def create_yolo_seg_label(
    crown_row,
    tile_transform,
    tile_geom,
    x_off: int,
    y_off: int,
    w: int,
    h: int,
    class_id: int,
) -> List[str] | None:
    """
    Convert a single tree-crown polygon into one or more YOLO segmentation
    annotation strings.

    The annotation format is:
        <class_id> x1 y1 x2 y2 ... xn yn
    where xi and yi are the normalised [0, 1] coordinates of each exterior
    vertex of the (possibly clipped) crown polygon.

    The crown is first intersected with the patch bounding box so that crowns
    crossing the patch edge are represented as partial polygons rather than
    being discarded entirely (unlike the detection task).  If the intersection
    is empty or geometrically invalid the function returns None.

    Args:
        crown_row:      Row from the tree-crown GeoDataFrame.
        tile_transform: Affine transform of the *patch* (not the full raster).
        tile_geom:      Shapely polygon representing the patch in map coordinates.
        x_off, y_off:   Top-left pixel offset of the patch within the raster
                        (unused here but kept for API consistency).
        w, h:           Patch width and height in pixels.
        class_id:       Integer class ID (0 for tree crown).

    Returns:
        List of YOLO segmentation label strings (one per polygon part), or None.
    """
    from shapely.geometry import Polygon

    geom      = crown_row.geometry
    geom_clip = geom.intersection(tile_geom)

    if geom_clip.is_empty or not geom_clip.is_valid:
        return None

    final_polys = []
    if geom_clip.geom_type == "Polygon":
        final_polys = [geom_clip]
    elif geom_clip.geom_type == "MultiPolygon":
        final_polys = list(geom_clip.geoms)
    elif geom_clip.geom_type == "GeometryCollection":
        final_polys = [p for p in geom_clip.geoms if isinstance(p, Polygon)]

    if not final_polys:
        return None

    yolo_annotations = []
    for poly in final_polys:
        poly_coords = []
        for x, y in poly.exterior.coords:
            px, py = ~tile_transform * (x, y)
            xn = min(1.0, max(0.0, px / w))
            yn = min(1.0, max(0.0, py / h))
            poly_coords.extend([xn, yn])

        if len(poly_coords) >= 6:
            poly_str = " ".join(f"{v:.6f}" for v in poly_coords)
            yolo_annotations.append(f"{class_id} {poly_str}")

    return yolo_annotations or None


**Save background patch (if allowed by split rules)**

In [ ]:
def save_background_if_allowed(
    src,
    window: Window,
    tile_id: str,
    images_dir: str,
    split_name: str,
) -> int:
    """
    Optionally save a background patch (no tree crowns inside) to disk.

    Background patches are excluded from val and test splits to ensure those
    splits are evaluated purely on positive examples.  For training a random
    fraction (BACKGROUND_RATIO) is kept to give the model examples of
    non-tree areas.

    Args:
        src:        Open rasterio DatasetReader.
        window:     Window describing the patch.
        tile_id:    Output filename stem.
        images_dir: Directory to write the GeoTIFF.
        split_name: 'train', 'val', or 'test'.

    Returns:
        0 (always).
    """
    if split_name in ("test", "val"):
        return 0
    if split_name == "train" and random.random() > BACKGROUND_RATIO:
        return 0

    tile_data      = src.read(window=window)
    tile_transform = rasterio.windows.transform(window, src.transform)
    patch_path     = os.path.join(images_dir, f"{tile_id}.tif")

    with rasterio.open(
        patch_path, "w",
        driver="GTiff",
        height=window.height,
        width=window.width,
        count=src.count,
        dtype=tile_data.dtype,
        crs=src.crs,
        transform=tile_transform,
    ) as dst:
        dst.write(tile_data)

    return 0


**Process one patch and write image + label files**

In [ ]:
def process_patch_and_annotate(
    src,
    crowns_vector,
    crowns_sindex,
    x_off: int,
    y_off: int,
    w: int,
    h: int,
    split_name: str,
    tile_id: str,
    images_dir: str,
    labels_dir: str,
) -> int:
    """
    Extract one image patch, generate YOLO segmentation labels for all
    intersecting tree crowns, and write image and label files.

    Unlike the detection task, crowns that cross the patch boundary are *not*
    discarded — they are clipped to the patch extent and the resulting partial
    polygon is annotated.  This gives the segmentation model more training signal
    at the cost of slightly inconsistent polygon shapes at patch edges.

    Args:
        src:            Open rasterio DatasetReader.
        crowns_vector:  Crown GeoDataFrame reprojected to the raster CRS.
        crowns_sindex:  Spatial index over crowns_vector.
        x_off, y_off:   Top-left pixel offset of the patch.
        w, h:           Patch dimensions in pixels.
        split_name:     'train', 'val', or 'test'.
        tile_id:        Output filename stem.
        images_dir:     Output directory for images.
        labels_dir:     Output directory for labels.

    Returns:
        Number of segmentation polygon annotations written.
    """
    window = Window(x_off, y_off, w, h)

    if x_off + w > src.width or y_off + h > src.height:
        return 0

    tile_geom = box(*rasterio.windows.bounds(window, src.transform))

    possible_ids   = list(crowns_sindex.intersection(tile_geom.bounds))
    possible_crowns = crowns_vector.iloc[possible_ids]
    crowns_intersect = possible_crowns[
        possible_crowns.geometry.intersects(tile_geom)
    ]

    if crowns_intersect.empty:
        return save_background_if_allowed(src, window, tile_id, images_dir, split_name)

    tile_data      = src.read(window=window)
    tile_transform = rasterio.windows.transform(window, src.transform)
    patch_path     = os.path.join(images_dir, f"{tile_id}.tif")
    label_path     = os.path.join(labels_dir, f"{tile_id}.txt")

    with rasterio.open(
        patch_path, "w",
        driver="GTiff",
        height=h, width=w,
        count=src.count,
        dtype=tile_data.dtype,
        crs=src.crs,
        transform=tile_transform,
    ) as dst:
        dst.write(tile_data)

    labels = []
    for _, row in crowns_intersect.iterrows():
        annot = create_yolo_seg_label(
            row, tile_transform, tile_geom, x_off, y_off, w, h, TREE_CLASS_ID
        )
        if annot:
            labels.extend(annot)

    if labels:
        with open(label_path, "w") as f:
            f.write("\n".join(labels))
        return len(labels)
    else:
        os.remove(patch_path)
        return 0


**Main loop: generate all patches and labels**

In [ ]:
def generate_patch_labels(
    raster_ids: List[str],
    crowns_gdf,
    split_mapping: Dict[str, str],
    splits_dir: str,
    tile_size: int,
) -> None:
    """
    Slide a window across each raster tile (held in memory via rasterio.MemoryFile)
    and call process_patch_and_annotate() for every position that lies fully
    within the raster bounds.

    Args:
        raster_ids:    List of tile IDs.
        crowns_gdf:    Tree-crown GeoDataFrame.
        split_mapping: Maps tile IDs to 'train', 'val', or 'test'.
        splits_dir:    Root output directory.
        tile_size:     Square patch side length in pixels.
    """
    for rid in raster_ids:
        rgb_path   = os.path.join(RGB_TILES_DIR, f"RGB_{rid}.tif")
        split_name = split_mapping[rid]

        overlap_percent = OVERLAP_BY_SPLIT[split_name]
        step_size       = int(tile_size * (1 - overlap_percent))

        images_dir = os.path.join(splits_dir, split_name, "images")
        labels_dir = os.path.join(splits_dir, split_name, "labels")
        os.makedirs(images_dir, exist_ok=True)
        os.makedirs(labels_dir, exist_ok=True)

        stacked, meta, src_obj = stack_rgb(rgb_path)
        src_obj.close()

        with rasterio.MemoryFile() as mem:
            with mem.open(**meta) as dataset:
                dataset.write(stacked)

            with mem.open() as src:
                width, height = src.width, src.height
                crowns_vector = crowns_gdf.to_crs(src.crs)
                crowns_sindex = crowns_vector.sindex

                n_cols = math.ceil((width  - tile_size) / step_size) + 1
                n_rows = math.ceil((height - tile_size) / step_size) + 1

                for i in range(n_rows):
                    for j in range(n_cols):
                        x_off   = j * step_size
                        y_off   = i * step_size
                        tile_id = f"RGB_{rid}_patch_{i}_{j}"

                        process_patch_and_annotate(
                            src, crowns_vector, crowns_sindex,
                            x_off, y_off, tile_size, tile_size,
                            split_name, tile_id, images_dir, labels_dir,
                        )


**Run**

In [ ]:
crowns_gdf = gpd.read_file(TREE_CROWNS_DIR)

generate_patch_labels(
    raster_ids    = RASTER_IDS,
    crowns_gdf    = crowns_gdf,
    split_mapping = SPLIT_MAP,
    splits_dir    = str(SPLITS_DIR),
    tile_size     = TILE_SIZE,
)
print("Patch generation complete.")
